In [1]:
import os
os.chdir(r"D:\product-analytics-ecommerce")
print(os.getcwd())

import duckdb
con = duckdb.connect("data/processed/project.duckdb")

D:\product-analytics-ecommerce


In [2]:
con.execute("""
CREATE OR REPLACE TABLE events_raw AS
SELECT * FROM read_csv_auto('data/raw/2019-Nov.csv')
""")
con.execute("SELECT COUNT(*) FROM events_raw").fetchall()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(67501979,)]

In [3]:
con.execute("""
CREATE OR REPLACE TABLE events AS
SELECT *
FROM events_raw
WHERE hash(user_id) % 10 = 0
""")
con.execute("SELECT COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM events").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,users
0,6726943,369828


In [4]:
con.execute("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT user_id) AS users,
  COUNT(DISTINCT user_session) AS sessions,
  MIN(event_time) AS first_event,
  MAX(event_time) AS last_event,
  SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_users,
  SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS null_category,
  SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) AS null_brand,
  SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) AS bad_price
FROM events
""").df()

,total_rows,users,sessions,first_event,last_event,null_users,null_category,null_brand,bad_price
0,6726943,369828,1370658,2019-11-01 00:00:03,2019-11-30 23:59:59,0.0,2171756.0,916447.0,18909.0


In [5]:
con.execute("""
SELECT event_type, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM events
GROUP BY event_type
ORDER BY n DESC
""").df()

,event_type,n,pct
0,view,6331304,94.12
1,cart,303954,4.52
2,purchase,91685,1.36


In [11]:
con.execute("""
SELECT COUNT(*) AS duplicate_rows FROM (
  SELECT event_time, event_type, product_id, user_id, user_session, COUNT(*) AS c
  FROM events
  GROUP BY ALL
  HAVING COUNT(*) > 1
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_rows
0,5791


In [7]:
import pandas as pd
cats = pd.read_csv("data/raw/cookie_cats.csv")
print(cats.shape)
cats.head()

(90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [9]:
con.close()

In [10]:
import os
os.chdir(r"D:\product-analytics-ecommerce")
import duckdb
con = duckdb.connect("data/processed/project.duckdb")

In [12]:
con.execute("""
CREATE OR REPLACE TABLE events_clean AS
SELECT DISTINCT *
FROM events
WHERE price > 0
""")
con.execute("""
SELECT
  (SELECT COUNT(*) FROM events) AS before_rows,
  (SELECT COUNT(*) FROM events_clean) AS after_rows
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,before_rows,after_rows
0,6726943,6697953


In [13]:
con.close()